# Cross-region coupling: does another region predict MEC firing beyond shared behaviour?

For a target neuron, does knowing the activity of neurons in a *different*
simultaneously recorded region improve prediction of its firing, over and above what
position and speed already explain?

**The baseline decides what the answer means.** Two spatially tuned, behaviourally
modulated populations will predict each other trivially through their shared inputs, so
the baseline is `position + speed`, not position alone:

    ΔpR² = pR²(position + speed + N cells from the other region) − pR²(position + speed)

**Pools, not single cells.** A single covariate neuron is sparse and makes the result
hostage to which cell was picked. Each target is predicted from a pool of `N_POOL`
neurons, drawn at random `N_REPEATS` times and averaged.

**Three conditions, and the comparison is between them, not against zero:**

| condition | covariate pool | role |
|---|---|---|
| `cross` | N cells from the partner region | the quantity of interest |
| `within` | N cells from the target's own region, **excluding its own shank** | the ceiling — cross-region coupling means little except relative to this |
| `shift` | the same cross-region pool, circularly shifted against the target | the floor |

The same-shank exclusion matters: spike-sorting bleed between nearby contacts
manufactures coupling that has nothing to do with circuitry.

`shift` replaces cerebellum as the negative control. The intended anatomical control has
only **3 sessions from 2 mice** (see the inventory below), which cannot carry a
specificity argument. A circular shift preserves each covariate's rate and
autocorrelation while destroying its timing relationship to the target, and is available
for every pair.

**Both directions** are run (MEC ← partner and partner ← MEC), since asymmetry is
informative.

In [ ]:
import sys, os, glob, re, time, itertools, warnings
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pynapple as nap
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'

SOURCE_PATH = '/Users/harryclark/Downloads/clark2025/'
FIGPATH = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
CACHE = '/Users/harryclark/Documents/spatial-manifolds/data/cross_region_xgboost'
os.makedirs(CACHE, exist_ok=True)

TIME_BS = 10            # ms bins, matching the established encoding-model config
HISTORY_MS = 1000       # raised-cosine history basis over the preceding second
TL = 200.0              # VR track length, cm
MIN_SPEED = 3.0
N_POOL = 10             # covariate pool size
N_REPEATS = 5           # random draws of the pool per target, averaged
CV_FOLDS = 10
CURATION = {'good', 'mua'}
MIN_SPIKES = 500

def region_group(r):
    """Fine-grained Allen labels -> the groupings this analysis compares."""
    r = str(r)
    if r.startswith('ENTm'): return 'MEC'
    if r.startswith('ENTl'): return 'LEC'
    if r.startswith('VIS'):  return 'VIS'
    if r.startswith('PAR'):  return 'PARA'
    if r.startswith(('PRE', 'POST')): return 'PRE'
    if r.startswith('SUB'):  return 'SUB'
    if r in ('root',):       return 'OTHER'
    return 'OTHER'

def session_cells(mouse, day):
    """Curated cells with region, shank and spike count for one VR session."""
    d = f'{SOURCE_PATH}M{mouse}/D{day}/VR/'
    stem = f'sub-M{mouse}_ses-D{day}_typ-VR'
    bp, cp = d + stem + '_beh.nwb', d + stem + '_srt-kilosort4_clusters.npz'
    if not (os.path.exists(bp) and os.path.exists(cp)):
        return None
    z = np.load(cp, allow_pickle=True)
    md_ = z['_metadata'].item()
    df = pd.DataFrame({k: np.asarray(md_[k]) for k in
                       ('brain_region', 'bombcell_label', 'coord_probe_x', 'coord_SCs_x')})
    df['cluster_id'] = np.asarray(z['keys'])
    df['region'] = df.brain_region.map(region_group)
    # shank from probe-plane x: 4 shanks, 250 um apart
    px = df.coord_probe_x.astype(float)
    df['shank'] = pd.qcut(px.rank(method='first'), 4, labels=False) if px.nunique() > 3 else 0
    df = df[df.bombcell_label.isin(CURATION)]
    return dict(beh_path=bp, clu_path=cp, cells=df)

print('setup ready')

## Which pairs can this be run on?

A fixed pool size is only meaningful if both regions can supply it. This is the
constraint that determines the scope of the whole analysis.

In [ ]:
inv_rows = []
for d in sorted(glob.glob(f'{SOURCE_PATH}M*/D*/VR')):
    m = re.search(r'M(\d+)/D(\d+)/VR', d)
    mouse, day = int(m.group(1)), int(m.group(2))
    s = session_cells(mouse, day)
    if s is None or not len(s['cells']):
        continue
    n = s['cells'].region.value_counts()
    inv_rows.append(dict(mouse=mouse, day=day, **{r: int(n.get(r, 0))
                          for r in ('MEC', 'VIS', 'PARA', 'LEC', 'PRE', 'SUB')}))
inv = pd.DataFrame(inv_rows)
print(f'{len(inv)} VR sessions with curated cells\n')
print('sessions where MEC and the partner region BOTH reach the pool size:')
for N in (5, 10, 20):
    row = {r: int(((inv[r] >= N) & (inv.MEC >= N)).sum()) for r in ('VIS', 'PARA', 'LEC', 'PRE', 'SUB')}
    print(f'   N={N:>2}: ' + '  '.join(f'{k} {v}' for k, v in row.items()))
usable = inv[(inv.VIS >= N_POOL) & (inv.MEC >= N_POOL)]
print(f'\nMEC-VIS usable at N={N_POOL}: {len(usable)} sessions, {usable.mouse.nunique()} mice')
display(usable[['mouse','day','MEC','VIS','PARA']].sort_values('VIS', ascending=False).head(12))

## The encoding model

Established configuration, unchanged: 10 ms bins, raised-cosine history basis over the
preceding 1000 ms, no spike history of the target (so predictions cannot be satisfied by
its own autocorrelation), contiguous 10-fold cross-validation, Poisson pseudo-R².

In [ ]:
from spatial_manifolds.mlencoding import MLencoding, poisson_pseudoR2

N_FILTERS = int(HISTORY_MS / TIME_BS)
def make_model():
    return MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                      window=TIME_BS, n_filters=N_FILTERS, max_time=HISTORY_MS)

def load_session(mouse, day):
    """Binned behaviour and spike counts on a common 10 ms base."""
    s = session_cells(mouse, day)
    beh = nap.load_file(s['beh_path']); clusters = nap.load_file(s['clu_path'])
    ids = list(clusters.index)
    last = clusters[ids[0]].count(bin_size=TIME_BS, time_units='ms').index[-1]
    ep = nap.IntervalSet(start=0, end=last, time_units='s')
    speed = np.asarray(beh['S'].bin_average(bin_size=TIME_BS, time_units='ms', ep=ep))
    trav = np.asarray(beh['travel'].bin_average(bin_size=TIME_BS, time_units='ms', ep=ep))
    trav = pd.Series(trav).ffill().bfill().values
    speed = pd.Series(speed).ffill().bfill().values
    pos = trav % TL
    counts = {c: np.asarray(clusters[c].count(bin_size=TIME_BS, time_units='ms', ep=ep))
              for c in ids}
    cells = s['cells'][s['cells'].cluster_id.isin(ids)].copy()
    cells['n_spikes'] = cells.cluster_id.map({c: counts[c].sum() for c in ids})
    cells = cells[cells.n_spikes >= MIN_SPIKES]
    return dict(pos=pos, speed=speed, counts=counts, cells=cells, n=len(pos))

def delta_pr2(target_y, base_X, pool_counts, model=None):
    """pR2 with the pool added, minus pR2 of the behavioural baseline."""
    model = model or make_model()
    _, pr2_base = model.fit_cv(base_X, target_y, verbose=0, continuous_folds=True)
    X = np.column_stack([base_X] + list(pool_counts))
    _, pr2_full = model.fit_cv(X, target_y, verbose=0, continuous_folds=True)
    return float(np.nanmean(pr2_full) - np.nanmean(pr2_base)), float(np.nanmean(pr2_base))

print(f'model: {N_FILTERS} history filters x {TIME_BS}ms = {HISTORY_MS}ms')

## Timing first

These are full-session fits with a 100-filter history basis, not the small
position-binned problems used elsewhere in this project. Measure the cost of one target
before committing to a sweep.

In [ ]:
def run_target(S, tgt, partner_region, rng, n_pool=N_POOL, n_repeats=N_REPEATS,
               conditions=('cross', 'within', 'shift')):
    """One target cell, all conditions, averaged over random pool draws."""
    y = S['counts'][tgt]
    base = np.column_stack([S['pos'], S['speed']])
    cells = S['cells']
    t_row = cells[cells.cluster_id == tgt].iloc[0]
    cross_pool = cells[cells.region == partner_region].cluster_id.values
    # exclude the target's own shank from the within-region pool: adjacent contacts
    # share spike-sorting errors, which manufactures coupling
    within_pool = cells[(cells.region == t_row.region) &
                        (cells.shank != t_row.shank) &
                        (cells.cluster_id != tgt)].cluster_id.values
    out = {}
    for cond in conditions:
        src = within_pool if cond == 'within' else cross_pool
        if len(src) < n_pool:
            out[cond] = np.nan; continue
        vals = []
        for _ in range(n_repeats):
            pick = rng.choice(src, n_pool, replace=False)
            cols = [S['counts'][c] for c in pick]
            if cond == 'shift':
                cols = [np.roll(c, rng.randint(len(c) // 10, len(c) - len(c) // 10))
                        for c in cols]
            d, _ = delta_pr2(y, base, cols)
            vals.append(d)
        out[cond] = float(np.nanmean(vals))
    return out

TIMING_SESSION = tuple(usable.sort_values('VIS', ascending=False).iloc[0][['mouse','day']])
_mo, _dy = int(TIMING_SESSION[0]), int(TIMING_SESSION[1])
print(f'timing on M{_mo} D{_dy} ...')
_t = time.time(); S = load_session(_mo, _dy)
print(f'  session loaded in {time.time()-_t:.0f}s: {S["n"]} bins, {len(S["cells"])} curated cells '
      f'({S["cells"].region.value_counts().to_dict()})')
_mec = S['cells'][S['cells'].region == 'MEC'].cluster_id.values
_t = time.time()
_res = run_target(S, _mec[0], 'VIS', np.random.RandomState(0), n_repeats=1)
_per_fit = (time.time() - _t) / 6      # 3 conditions x (baseline + full)
print(f'  one target, 1 repeat, 3 conditions: {time.time()-_t:.0f}s  (~{_per_fit:.1f}s per fit)')
print(f'  -> {_res}')
print(f'\nprojected: {len(usable)} sessions x 30 targets x 2 directions x 3 conditions x '
      f'{N_REPEATS} repeats x 2 fits = {len(usable)*30*2*3*N_REPEATS*2:,} fits '
      f'= {len(usable)*30*2*3*N_REPEATS*2*_per_fit/3600:.0f} h single-core')

## Scoped run

Scale the sweep to what the timing supports. Targets are subsampled per session, and
statistics are computed at session level — cells recorded together share the animal's
behaviour, so pooling them across a session would treat one session as many independent
observations.

In [ ]:
RUN = False        # set True to launch; see the projection above first
MAX_TARGETS = 20   # per session per direction

if RUN:
    rows, t0 = [], time.time()
    for k, r in enumerate(usable.itertuples(), 1):
        try:
            S = load_session(int(r.mouse), int(r.day))
        except Exception as e:
            print(f'  ! M{r.mouse} D{r.day}: {type(e).__name__}'); continue
        rng = np.random.RandomState(int(r.mouse) * 100 + int(r.day))
        for tgt_region, partner in (('MEC', 'VIS'), ('VIS', 'MEC')):
            pool = S['cells'][S['cells'].region == tgt_region].cluster_id.values
            if len(pool) < 2: continue
            for tgt in rng.permutation(pool)[:MAX_TARGETS]:
                res = run_target(S, tgt, partner, rng)
                rows.append(dict(mouse=int(r.mouse), day=int(r.day), target=int(tgt),
                                 target_region=tgt_region, partner=partner, **res))
        print(f'[{k}/{len(usable)}] M{r.mouse} D{r.day} — {len(rows)} targets, '
              f'{time.time()-t0:.0f}s', flush=True)
    res_df = pd.DataFrame(rows)
    res_df.to_csv(f'{CACHE}/cross_region_deltapr2.csv', index=False)
else:
    p = f'{CACHE}/cross_region_deltapr2.csv'
    res_df = pd.read_csv(p) if os.path.exists(p) else pd.DataFrame()
    print('RUN=False — set True to compute' if res_df.empty else f'loaded {len(res_df)} targets')

## Result

The question is not whether `cross` exceeds zero — it is where `cross` sits between the
`shift` floor and the `within` ceiling.

In [ ]:
if len(res_df):
    from scipy.stats import wilcoxon
    S_ = res_df.groupby(['mouse','day','target_region','partner'])[
            ['cross','within','shift']].median().reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for ax, (tr, gg) in zip(axes, S_.groupby('target_region')):
        for i, c in enumerate(['shift','cross','within']):
            v = gg[c].dropna()
            ax.scatter(np.full(len(v), i) + np.random.uniform(-.1,.1,len(v)), v, s=14,
                       color=['0.6', '#c04744', '#2171b5'][i], alpha=.75)
            ax.plot([i-.25, i+.25], [v.median()]*2, color='k', lw=1.6)
        ax.axhline(0, color='k', ls=':', lw=.8)
        ax.set_xticks(range(3)); ax.set_xticklabels(['shift\n(floor)','cross','within\n(ceiling)'])
        ax.set_ylabel('ΔpR² over position+speed')
        ax.set_title(f'target {tr}  ←  {gg.partner.iloc[0]}', fontsize=9)
        ax.spines[['top','right']].set_visible(False)
    plt.tight_layout(); plt.savefig(f'{FIGPATH}/cross_region_coupling.pdf', dpi=200, bbox_inches='tight')
    plt.show()
    for (tr, pa), gg in S_.groupby(['target_region','partner']):
        d = gg.dropna(subset=['cross','shift'])
        if len(d) > 5:
            st, p = wilcoxon(d.cross, d.shift)
            frac = (d.cross / d.within.replace(0, np.nan)).median()
            print(f'{tr} <- {pa}: n={len(d)} sessions | cross {d.cross.median():+.4f} vs '
                  f'shift {d.shift.median():+.4f} (p={p:.3g}) | cross/within = {frac:.2f}')